# 数据探索笔记

用于探索和分析 A 股股票数据

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_collector.data_manager import data_manager
from src.data_collector.tushare_client import TushareClient
from src.utils.database import init_db

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [ ]:
# 初始化数据库
init_db()

In [ ]:
# 获取股票列表
ts_client = TushareClient()
stocks = ts_client.get_stock_list(list_status='L')
print(f"A 股上市公司数量：{len(stocks)}")
stocks.head()

In [ ]:
# 按行业统计
industry_count = stocks['industry'].value_counts().head(20)
plt.figure(figsize=(12, 8))
industry_count.plot(kind='barh')
plt.title('按行业统计上市公司数量')
plt.xlabel('公司数量')
plt.tight_layout()
plt.show()

In [ ]:
# 获取贵州茅台的行情数据
mt_df = data_manager.get_daily_quotes('600519.SH')
print(f"贵州茅台数据量：{len(mt_df)} 条")
mt_df.head()

In [ ]:
# 绘制股价走势
if not mt_df.empty:
    mt_df['trade_date'] = pd.to_datetime(mt_df['trade_date'])
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(mt_df['trade_date'], mt_df['close'], label='收盘价')
    ax.set_title('贵州茅台 (600519.SH) 股价走势')
    ax.set_xlabel('日期')
    ax.set_ylabel('价格 (元)')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# 计算技术指标
from src.utils.helpers import calculate_ma, calculate_macd, calculate_rsi

if not mt_df.empty:
    close = mt_df['close']
    
    # 均线
    ma_data = calculate_ma(close, [5, 10, 20, 60])
    
    # MACD
    macd_data = calculate_macd(close, 12, 26, 9)
    
    # RSI
    rsi = calculate_rsi(close, 14)
    
    # 绘制
    fig, axes = plt.subplots(4, 1, figsize=(14, 12))
    
    # 股价和均线
    axes[0].plot(mt_df['trade_date'], close, label='收盘价')
    for col in ma_data.columns:
        axes[0].plot(mt_df['trade_date'], ma_data[col], label=col, alpha=0.7)
    axes[0].set_title('股价和均线')
    axes[0].legend()
    
    # MACD
    axes[1].plot(mt_df['trade_date'], macd_data['dif'], label='DIF')
    axes[1].plot(mt_df['trade_date'], macd_data['dea'], label='DEA')
    axes[1].bar(mt_df['trade_date'], macd_data['macd'], label='MACD', alpha=0.5)
    axes[1].set_title('MACD 指标')
    axes[1].legend()
    
    # RSI
    axes[2].plot(mt_df['trade_date'], rsi, label='RSI(14)')
    axes[2].axhline(y=70, color='r', linestyle='--', alpha=0.5, label='超买线')
    axes[2].axhline(y=30, color='g', linestyle='--', alpha=0.5, label='超卖线')
    axes[2].set_title('RSI 指标')
    axes[2].legend()
    axes[2].set_ylim(0, 100)
    
    # 成交量
    axes[3].bar(mt_df['trade_date'], mt_df['vol'] / 1e6, label='成交量')
    axes[3].set_title('成交量 (百万股)')
    axes[3].legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
# 获取财务数据
financial_df = data_manager.get_financial_indicators('600519.SH')
if not financial_df.empty:
    print("贵州茅台财务指标:")
    print(financial_df[['ann_date', 'pe', 'pb', 'roe', 'net_margin']].head(10))

In [ ]:
# 对比多只股票
stock_codes = ['600519.SH', '000858.SZ', '000568.SZ']  # 茅台、五粮液、泸州老窖
stock_names = ['贵州茅台', '五粮液', '泸州老窖']

fig, ax = plt.subplots(figsize=(14, 6))

for code, name in zip(stock_codes, stock_names):
    df = data_manager.get_daily_quotes(code)
    if not df.empty:
        df['trade_date'] = pd.to_datetime(df['trade_date'])
        # 归一化到基准 100
        base_price = df.iloc[0]['close']
        df['normalized'] = df['close'] / base_price * 100
        ax.plot(df['trade_date'], df['normalized'], label=name)

ax.set_title('白酒股走势对比 (归一化)')
ax.set_xlabel('日期')
ax.set_ylabel('相对价格')
ax.legend()
plt.tight_layout()
plt.show()